# Lesson 6 — The exchange publishes its own logical structure

Mutual exclusivity, strike ladders, buckets and settlement sources are all in the metadata. Two markets are the same payoff only when they share a settlement source — never when their titles merely read alike.

**The rule.** `A ⊆ B ⟹ p(A) ≤ p(B)`

**When it holds.** Within one event, where every child market shares a shard and settles on one source.

**When it fails.** Matching markets by title similarity. Two similarly-worded markets can settle on different sources with different cut-offs, and when they resolve differently a hedged position pays zero or two dollars rather than one.

| | |
|---|---|
| Lesson id | `lattice` |
| Pane it appears on | `lattice` (panes carry more than one lesson) |
| Code it is about | `modules/coherence/kernel/lattice.py` |
| Tests that go red if it stops being true | `tests/test_coherence_lattice.py` |
| Pane shipped | yes |

Every cell below runs against the real kernel. Nothing here is a re-implementation:
a number this notebook prints is the number the engine would produce for the same
input. The recorded Kalshi payloads come from `tests/fixtures/coherence/`.

In [ ]:
import json
import sys
from decimal import Decimal
from pathlib import Path

# This notebook lives in notebooks/coherence_lab/ and imports the kernel two
# levels up. Found by walking upward rather than by counting parents, so the
# notebook runs from its own directory or from Part2_Infrastructure.
HERE = Path.cwd().resolve()
ROOT = next((path for path in (HERE, *HERE.parents) if (path / "modules" / "coherence" / "kernel").is_dir()), None)
if ROOT is None:
    raise SystemExit(f"no coherence kernel above {HERE}: open this notebook from inside Part2_Infrastructure")
sys.path.insert(0, str(ROOT))

FIXTURES = ROOT / "tests" / "fixtures" / "coherence"


def fixture(name: str) -> dict:
    """One recorded Kalshi response, envelope and all, exactly as it was sent.

    These are captures, not mocks. Where a number below looks odd it is because
    the exchange quoted it, and `tools/capture_kalshi_fixtures.py` re-records
    them.
    """
    return json.loads((FIXTURES / f"{name}.json").read_text(encoding="utf-8"))


print(f"kernel root       {ROOT}")
print(f"recorded fixtures {FIXTURES.is_dir()}")

## 1. A recorded ladder, and the implications the venue's own metadata carries

In [ ]:
from modules.coherence.drivers.kalshi_parse import Event, parse_event, parse_market
from modules.coherence.kernel import distribution
from modules.coherence.kernel.book import Book, Level
from modules.coherence.kernel.lattice import Component, Node, build_component
from modules.coherence.kernel.states import build_states

crypto = [parse_market(row, "KXBTCD") for row in fixture("markets_crypto")["body"]["markets"]]
ladder_event = Event(
    event_ticker=crypto[0].event_ticker,
    series_ticker="KXBTCD",
    title="BTC daily strike ladder, from a recorded /markets payload",
    # The /markets route carries no exclusivity flag. False here because the
    # venue did not say otherwise, not because we decided it.
    mutually_exclusive=False,
    exchange_index=crypto[0].exchange_index,
    settlement_sources=(),
    markets=tuple(crypto),
)
ladder = build_component(ladder_event)

print(f"  {len(ladder.nodes)} rungs, {len(ladder.edges)} edges, scope {ladder.scope}")
print()
for edge in ladder.edges[:2]:
    print(f"  {edge.kind}: {edge.source} -> {edge.target}")
    print(f"    {edge.because}")
print()
print("  Only ADJACENT strikes are linked. The relation is transitive, so every")
print("  non-adjacent pair is implied by the chain and emitting them all would inflate")
print("  the matrix without adding a single constraint.")

## 2. Settlement sources are the equivalence test

In [ ]:
fed = parse_event(fixture("event_mee")["body"])
print(f"  {fed.event_ticker} settles on {fed.settlement_sources}")
print(f"  the recorded /markets payloads carry no source at all: {ladder_event.settlement_sources}")
print()

# Two markets whose titles read alike, settling on different sources.
divergent = Component(
    component_id="LOOKALIKE",
    event_ticker="LOOKALIKE",
    series_ticker="LOOKALIKE",
    exchange_index=0,
    mutually_exclusive=False,
    nodes=[
        Node("A", "EA", "LOOKALIKE", 0, "custom", None, None, ("Source One",), "Highest temperature in NYC"),
        Node("B", "EB", "LOOKALIKE", 0, "custom", None, None, ("Source Two",), "NYC high temperature"),
    ],
)
sources = {node.settlement_sources for node in divergent.nodes}
print(f"  two lookalike titles, {len(sources)} distinct settlement sources: {sorted(sources)}")
print("  same payoff? The test is source equality, never title similarity. On the day")
print("  they disagree a 'hedged' position pays zero or two dollars rather than one.")

## 3. From structure to states of the world

In [ ]:
temps = [parse_market(row, "KXHIGHNY") for row in fixture("markets_ladder")["body"]["markets"]]
weather_event = Event(
    event_ticker=temps[0].event_ticker,
    series_ticker="KXHIGHNY",
    title="NYC high temperature, from a recorded /markets payload",
    mutually_exclusive=False,
    exchange_index=0,
    settlement_sources=(),
    markets=tuple(temps),
)
weather = build_component(weather_event)
space = build_states(weather)

print(f"  {space.note}")
print()
print("  market                  " + "  ".join(f"{state:>16}" for state in space.states))
for label, row in zip(space.labels, space.payoff, strict=True):
    print(f"  {label:<22}  " + "  ".join(f"{value:>16}" for value in row))
print()
print("  Three of these intervals are states no market pays in, because the underlying is")
print("  whole degrees and the listed buckets are 80-81, 82-83, 84-85. That is why the")
print("  exclusivity FLAG beats our inference from two strike numbers when it is present.")

## 4. The distribution those prices imply

In [ ]:
weather_books = {market.ticker: market.top for market in temps}
surface = distribution.build_surface(weather, weather_books)

print(f"  engine {surface.engine} on the {surface.basis} side; {surface.detail}")
print()
for item in surface.bins:
    print(f"    {item.label:<26} {item.mass}")
print()
print(f"  total mass       {surface.total_mass}")
print(f"  tail below       {surface.tail_mass_low}")
print(f"  tail above       {surface.tail_mass_high}")
print(f"  moments          {surface.mean!r}")
print(f"  why              {surface.moments_note}")

## 5. A survival curve that rises, and the bin below the axis

In [ ]:
# A ladder quoted so that a higher strike is DEARER than a lower one. The
# subtraction shows it as a bin below the axis; constraints.py prices the same
# fault as a monotone violation.
STRIKES = (("100000", "0.6000", "0.3800"), ("105000", "0.6400", "0.3400"), ("110000", "0.2000", "0.7800"))
rungs = [
    Node(f"T{strike}", "SYN", "SYN", 0, "greater", Decimal(strike), None, ("synthetic",), f"above {strike}")
    for strike, _yes, _no in STRIKES
]
inverted = Component(
    component_id="SYN", event_ticker="SYN", series_ticker="SYN",
    exchange_index=0, mutually_exclusive=False, nodes=rungs,
)
inverted_books = {
    node.ticker: Book(
        ticker=node.ticker,
        yes_bids=(Level(Decimal(yes_bid), 10_000),),
        no_bids=(Level(Decimal(no_bid), 10_000),),
    )
    for node, (_strike, yes_bid, no_bid) in zip(rungs, STRIKES, strict=True)
}

bad = distribution.build_surface(inverted, inverted_books)
for item in bad.bins:
    flag = "   NEGATIVE MASS" if item.is_negative else ""
    print(f"    {item.label:<26} {item.mass}{flag}")
print()
print(f"  negative bins: {bad.negative_bins}")
print(f"  total mass still {bad.total_mass}, which is why totalling the pmf cannot find this")